# 04 - Governed Text-to-SQL Agent

This notebook shows the canonical AI analyst pattern: translate a business question into SQL, but route the plan through Metatate before returning anything executable.

The agent intentionally starts with an over-broad draft that includes a direct identifier. Metatate returns a conditional decision and the agent revises the query to a safer aggregate.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Business question


In [ ]:
question = "Which active customer segments have the most ARR by region?"
table_name = "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS"
print(question)


## Discover governed context before drafting SQL


In [ ]:
context = client.get_decision_context(table_name)
meaning = client.inspect_data_meaning(table_name)
rules = client.inspect_governance_rules(table_name)

print("Policy summary")
print(json.dumps(context["data"]["policy_summary"], indent=2))
print("\nColumns")
print(pd.DataFrame(meaning["data"]["columns"])[
    ["column_name", "data_category", "is_pii", "effective_sensitivity", "masking_type"]
].to_string(index=False))


## Draft, validate, and revise


In [ ]:
draft_sql = (
    "SELECT region, customer_name, email, SUM(arr) AS arr "
    "FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS "
    "WHERE account_status = 'active' "
    "GROUP BY region, customer_name, email"
)

validation = client.validate_query_context(
    draft_sql,
    operation="read",
    intended_use="analytics",
    actor_role="DATA_ANALYST",
)

print("Draft SQL")
print(draft_sql)
print("\nMetatate validation")
print(json.dumps(validation["data"]["decision"], indent=2))
print("\nFindings")
print(pd.DataFrame(validation["data"].get("sql_findings", [])))


In [ ]:
findings = validation["data"].get("sql_findings", [])
needs_revision = any(
    item.get("code") == "PII_COLUMN_SELECTED"
    or item.get("type") == "PII_COLUMN"
    or (item.get("code") == "MASKING_REQUIRED" and item.get("column"))
    for item in findings
)

if needs_revision:
    final_sql = (
        "SELECT region, account_status, SUM(arr) AS arr "
        "FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS "
        "WHERE account_status = 'active' "
        "GROUP BY region, account_status"
    )
    final_validation = client.validate_query_context(
        final_sql,
        operation="read",
        intended_use="analytics",
        actor_role="DATA_ANALYST",
    )
else:
    final_sql = draft_sql
    final_validation = validation

print("Final SQL")
print(final_sql)
print("\nFinal decision")
print(json.dumps(final_validation["data"]["decision"], indent=2))
print("\nAgent response")
print(final_validation["data"]["agent_action"]["message"])


The agent is useful because it can revise. The decision layer does not just say no; it gives the agent enough context to remove risky fields and continue with an allowed analytical answer.
